<a href="https://colab.research.google.com/github/amaimanwar8-arch/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/amaimanwar8-arch/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Working dir:", os.getcwd())
print(df.shape[0], "pages loaded successfully.")

Working dir: /content/flyrank-ml-internship
30000 pages loaded successfully.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [2]:
print("""
Task type: Ranking / scoring (supervised learning).

My lane's question is "which pages should be reviewed first" - a "which ones first?"
question, which the framing-ml-problems skill maps to Ranking/scoring, with a priority
score as the target and precision@K as the metric. This is SUPERVISED, not unsupervised:
I have an observed target (a proxy label, see Section 2) to train against, unlike
clustering (Lane 3), which would have no target at all and just group similar pages.
""")


Task type: Ranking / scoring (supervised learning).

My lane's question is "which pages should be reviewed first" - a "which ones first?"
question, which the framing-ml-problems skill maps to Ranking/scoring, with a priority
score as the target and precision@K as the metric. This is SUPERVISED, not unsupervised:
I have an observed target (a proxy label, see Section 2) to train against, unlike
clustering (Lane 3), which would have no target at all and just group similar pages.



## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [3]:
print("""
Target/proxy: is_declining_label = (trend_direction == "down"), same as the starter
pipeline.

Important caveat from the flyrank-data skill: trend_direction is ITSELF derived from
trend_pct. That means trend_direction and trend_pct can NEVER be used as features later
- only as the source of this target. If I fed them back in as features, the model would
just be reading its own answer off a different column - a leakage trap, not a discovery.

This target is also a PROXY, not a true future outcome - it's a bucket from the CURRENT
window, not something observed after a decision point. A stronger capstone target would
be a future-window label (prior 90 days -> decline over the next 30 days).
""")


Target/proxy: is_declining_label = (trend_direction == "down"), same as the starter
pipeline.

Important caveat from the flyrank-data skill: trend_direction is ITSELF derived from
trend_pct. That means trend_direction and trend_pct can NEVER be used as features later
- only as the source of this target. If I fed them back in as features, the model would
just be reading its own answer off a different column - a leakage trap, not a discovery.

This target is also a PROXY, not a true future outcome - it's a bucket from the CURRENT
window, not something observed after a decision point. A stronger capstone target would
be a future-window label (prior 90 days -> decline over the next 30 days).



## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [4]:
print("""
Success metric: Precision@50.

This matches the real decision constraint: a reviewer can only act on a limited queue,
not the whole page inventory. Precision@50 asks "of the top 50 pages ranked for review,
how many were actually worth it" - which is the same metric the starter pipeline reports
(baseline 0.240 vs. random forest 0.740). I'm also tracking Precision@20 as a secondary
check, since Notebook 1 showed the hand-rule actually wins at @20 even though the model
wins at @50 - a reminder that the metric choice changes which method "wins."
""")


Success metric: Precision@50.

This matches the real decision constraint: a reviewer can only act on a limited queue,
not the whole page inventory. Precision@50 asks "of the top 50 pages ranked for review,
how many were actually worth it" - which is the same metric the starter pipeline reports
(baseline 0.240 vs. random forest 0.740). I'm also tracking Precision@20 as a secondary
check, since Notebook 1 showed the hand-rule actually wins at @20 even though the model
wins at @50 - a reminder that the metric choice changes which method "wins."



## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [5]:
# avg_position = 0 means "no data," not rank zero - exclude those rows first
clean = df[df["avg_position"] > 0].copy()

target = clean["trend_direction"].str.lower().eq("down").astype(int)

df_view = clean[["content_id", "impressions_90d", "days_since_last_update",
                  "avg_position", "ctr", "word_count"]].copy()
df_view["target_is_declining"] = target

print("One row = one page (content_id). avg_position=0 rows excluded (no real position data).")
print("Note: ctr is a x100-scaled rate column (0.76 means 0.76%, not 76%) - shown as-is, not rescaled.")
df_view.head(10)

One row = one page (content_id). avg_position=0 rows excluded (no real position data).
Note: ctr is a x100-scaled rate column (0.76 means 0.76%, not 76%) - shown as-is, not rescaled.


,content_id,impressions_90d,days_since_last_update,avg_position,ctr,word_count,target_is_declining
0,content_304f48230142,3803,20,10.6,0.76,3221.0,1
1,content_a1fb4e703a9e,15320,25,20.3,0.05,2481.0,1
2,content_9aa793d4d895,12581,20,36.5,0.09,3515.0,1
3,content_331d6c4de07b,11751,22,6.2,0.49,NaN,0
4,content_d99b7a2d90ca,19140,14,44.0,0.13,2803.0,1
5,content_d4084a4bc775,3970,20,8.5,0.03,3080.0,1
6,content_9a34b442b552,20,20,7.0,0.00,3059.0,1
7,content_a63219c6e95a,1724,22,21.2,0.06,NaN,0
8,content_5e6c160719bc,32574,20,46.0,0.09,3807.0,1
9,content_c27558df2b0c,1240,104,4.9,0.16,NaN,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [6]:
print("""
A fixed rule can combine maybe 2-3 signals before a human can no longer read or trust
it - that's literally what Notebook 2 showed: a depth-2 tree is readable, but by depth-4
it's already too branchy to sanity-check by eye. Real pages have many interacting
signals (position, freshness, CTR, word count, trend) that don't reduce cleanly to a
short if/else chain. The starter pipeline's own numbers make the case directly: the
hand-written baseline hits Precision@50 = 0.240, while a learned model reaches 0.740 -
about 3x more of the top 50 flagged pages are actually correct. That gap is the evidence
that ML earns its place here rather than being complexity for its own sake.
""")


A fixed rule can combine maybe 2-3 signals before a human can no longer read or trust
it - that's literally what Notebook 2 showed: a depth-2 tree is readable, but by depth-4
it's already too branchy to sanity-check by eye. Real pages have many interacting
signals (position, freshness, CTR, word count, trend) that don't reduce cleanly to a
short if/else chain. The starter pipeline's own numbers make the case directly: the
hand-written baseline hits Precision@50 = 0.240, while a learned model reaches 0.740 -
about 3x more of the top 50 flagged pages are actually correct. That gap is the evidence
that ML earns its place here rather than being complexity for its own sake.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.